# YZM358 - Code Review Chatbot (Üye 2)
Bu notebook, modeli sıfırdan pad-token (-100) maskelemesi ile doğru şekilde eğitmenizi ve RAG testini yapmanızı sağlar.

In [ ]:
!pip install --upgrade transformers datasets faiss-cpu sacrebleu bert-score sentencepiece

## ADIM 1: Temizlenmiş (Hazır) Veriyi Yükleme
Burada kendi temizlediğin veriyi nasıl içeri alacağını görüyorsun.

In [ ]:
from google.colab import drive
from datasets import load_dataset

# 1. Drive'ı bağla
drive.mount('/content/drive')

# 👉 ÖNEMLİ ADIM:
# "CodeReviewBot" klasörü "Benimle Paylaşılanlar"da olduğu için Colab bunu bazen doğrudan göremez.
# Drive'ına girip "CodeReviewBot" klasörüne sağ tıkla -> "Drive'ıma Kısayol Ekle" (Add shortcut to Drive) de.

dosya_yolu = "/content/drive/MyDrive/CodeReviewBot/data/codereviewer_clean/train_clean.jsonl"

print("Veri yükleniyor... (1.44 GB)")
dataset = load_dataset("json", data_files=dosya_yolu)
senin_train_verin = dataset["train"]


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Veri yükleniyor... (1.44 GB)


In [ ]:
from huggingface_hub import snapshot_download
# Hızlı sürüme ait bozuk dosyayı (tokenizer.json) görmezden gelerek sadece sağlam dosyaları lokalimize indiriyoruz!
snapshot_download(repo_id="Salesforce/codet5p-220m", local_dir="./temiz_tokenizer", ignore_patterns=["tokenizer.json"])


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

'/content/temiz_tokenizer'

## ADIM 2: Modelin Ana Sınıfı (Beyin Kısmı)
Bu hücreyi direkt çalıştır geç. İçinde `-100` maskelemesi olan doğru eğitim kodu var.

In [ ]:
import torch
import faiss
import numpy as np
import os
import shutil
from transformers import RobertaTokenizer, T5ForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
from huggingface_hub import snapshot_download

# --- 1. BOZUK TOKENIZER'I FİZİKSEL OLARAK TAMİR ETME ---
print("Tokenizer dosyaları indiriliyor...")
os.makedirs("./temiz_tokenizer", exist_ok=True)
snapshot_download(repo_id="Salesforce/codet5p-220m", local_dir="./temiz_tokenizer")

for bad_file in ["special_tokens_map.json", "tokenizer_config.json", "added_tokens.json", "tokenizer.json"]:
    bad_path = f"./temiz_tokenizer/{bad_file}"
    if os.path.exists(bad_path):
        os.remove(bad_path)
print("Bozuk ayar dosyaları silindi! Tokenizer sadece saf sözlükten hatasız oluşturulacak.")


class CodeReviewModelRAG:
    def __init__(self, model_name_or_path="Salesforce/codet5p-220m", device="cuda" if torch.cuda.is_available() else "cpu"):
        self.device = device
        self.model_name = model_name_or_path

        self.tokenizer = RobertaTokenizer.from_pretrained("./temiz_tokenizer")
        self.model = T5ForConditionalGeneration.from_pretrained(self.model_name).to(device)
        self.index = None
        self.corpus_embeddings = []
        self.corpus_data = []
        print(f"[{self.device.upper()}] Model {self.model_name} başarıyla yüklendi!")

    def _get_embedding(self, text):
        inputs = self.tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512).to(self.device)
        with torch.no_grad():
            outputs = self.model.encoder(**inputs)
            embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
        return embeddings

    def build_faiss_index(self, dataset, column_name="input"):
        print(f"FAISS indeksi oluşturuluyor... (Veri boyutu: {len(dataset)})")
        subset = dataset.select(range(min(2000, len(dataset))))
        embeddings_list = []
        for i, item in enumerate(subset):
            text = item[column_name]
            emb = self._get_embedding(text)
            embeddings_list.append(emb)
            self.corpus_data.append(item)

        self.corpus_embeddings = np.vstack(embeddings_list).astype("float32")
        self.index = faiss.IndexFlatL2(self.corpus_embeddings.shape[1])
        self.index.add(self.corpus_embeddings)
        print("FAISS indeksi başarıyla oluşturuldu!")

    def retrieve_similar_examples(self, query_code, k=2):
        if self.index is None: return []
        query_emb = self._get_embedding(query_code).astype("float32")
        distances, indices = self.index.search(query_emb, k)
        return [self.corpus_data[idx] for idx in indices[0] if idx != -1]

    def generate_review(self, query_code):
        similar_examples = self.retrieve_similar_examples(query_code, k=2)
        prompt = "Review this code:\n"
        if similar_examples:
            prompt += "Context (Similar Examples):\n"
            for i, ex in enumerate(similar_examples):
                prompt += f"Example {i+1} Review: {ex.get('msg', '')}\n"

        prompt += f"\nCode to review:\n{query_code}\n\nReview:"
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(self.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_length=256,
                min_length=5,
                num_beams=4,
                no_repeat_ngram_size=3,
                early_stopping=True
            )
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

    def train_model(self, train_dataset, eval_dataset, output_dir="./results", epochs=3, batch_size=4):
        print("Model eğitimi (Fine-Tuning) başlıyor...")

        def preprocess_function(examples):
            inputs = ["Review this code: " + str(code) for code in examples["input"]]
            targets = [str(review) for review in examples["msg"]]

            # ESKİ VE HATA VEREN as_target_tokenizer() KISMI TAMAMEN SİLİNDİ, YERİNE text_target EKLENDİ!
            model_inputs = self.tokenizer(inputs, max_length=512, padding="max_length", truncation=True)
            labels = self.tokenizer(text_target=targets, max_length=128, padding="max_length", truncation=True)

            labels["input_ids"] = [
                [(l if l != self.tokenizer.pad_token_id else -100) for l in label]
                for label in labels["input_ids"]
            ]
            model_inputs["labels"] = labels["input_ids"]
            return model_inputs

        tokenized_train = train_dataset.map(preprocess_function, batched=True)
        tokenized_eval = eval_dataset.map(preprocess_function, batched=True)

        training_args = Seq2SeqTrainingArguments(
            output_dir=output_dir,
            eval_strategy="epoch",
            learning_rate=2e-5,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            weight_decay=0.01,
            save_total_limit=2,
            num_train_epochs=epochs,
            predict_with_generate=True,
            fp16=False
        )

        trainer = Seq2SeqTrainer(
            model=self.model,
            args=training_args,
            train_dataset=tokenized_train,
            eval_dataset=tokenized_eval,
            processing_class=self.tokenizer,
            data_collator=DataCollatorForSeq2Seq(self.tokenizer, model=self.model),
        )

        trainer.train()
        self.model.save_pretrained(os.path.join(output_dir, "final_model"))
        self.tokenizer.save_pretrained(os.path.join(output_dir, "final_model"))
        print("Eğitim tamamlandı ve model kaydedildi!")


Tokenizer dosyaları indiriliyor...


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Bozuk ayar dosyaları silindi! Tokenizer sadece saf sözlükten hatasız oluşturulacak.


In [ ]:
print(senin_train_verin.column_names)


['input', 'oldf', 'msg', 'y']


## ADIM 3: Modeli Başlatma ve Eğitime Sokma
Artık modeli internetten taze taze indirip, kendi verimizle doğru şekilde eğitiyoruz.

In [ ]:
# 1. Modeli taze ağırlıklarıyla kur
rag_system = CodeReviewModelRAG()

# 2. Eğitimi Başlat
rag_system.train_model(
    train_dataset=senin_train_verin,
    eval_dataset=senin_train_verin.select(range(1000)), # Test verisi olarak eğitimin ilk 1000 satırını kullanıyoruz
    epochs=3,
    batch_size=4
)

print("Eğitim Bitti!")


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

[CUDA] Model Salesforce/codet5p-220m başarıyla yüklendi!
Model eğitimi (Fine-Tuning) başlıyor...


Map:   0%|          | 0/48874 [00:00<?, ? examples/s]

[transformers] Error during conversion: ReadTimeout('The read operation timed out')
Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 117, in auto_conversion
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 96, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 77, in get_conversion_pr_reference
    raise OSError(
OSError: Could not create safetensors conversion PR. T

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Eğitim tamamlandı ve model kaydedildi!
Eğitim Bitti!


## ADIM 4: FAISS RAG Altyapısı ve Test
Eğitim bittikten sonra FAISS indeksini oluşturup modelden kod incelemesi istiyoruz.

In [ ]:
# 1. FAISS veri tabanını oluştur (Benzer kod örnekleri bulabilmesi için)
rag_system.build_faiss_index(senin_train_verin, column_name="code")

# 2. Test etmek istediğin kodu yaz
test_kodu = """
def divide(a, b):
    return a / b
"""

# 3. Modelden yorum üretmesini iste!
sonuc = rag_system.generate_review(test_kodu)
print("\n--- MODELİN İNCELEMESİ (REVIEW) ---")
print(sonuc)


FAISS indeksi oluşturuluyor... (Veri boyutu: 48874)


KeyError: 'code'